# Multi-Video Analysis — 28 debates × 8 candidates

This notebook aggregates findings across the **full corpus** (28 debates, ~2600 audio
segments, ~1000+ visual segments from `seg_all`).

**Structure**
- **Part I — Audio-only** acoustic EDA, candidate profiles, transcript analysis,
  within-debate dynamics (arousal + entrainment). Mirrors the content removed from
  `labeling_audio/correlations_audio.ipynb`.
- **Part II — Audio ↔ Visual** at multi-video scale: Sections **B, C, D** from
  `labeling_visual/corelations_updated.ipynb`, now with 1000+ segments and all 8
  candidates. Uses the same section letters so your colleague can directly recognise
  her work scaled up.
- **Part III — New findings** only possible across all debates: candidate emotion
  profiles, poll correlation, opponent effect, joint dimensionality reduction.

## Setup

In [ ]:
import sys, os
# works whether Jupyter is started from project root or from a subfolder
sys.path.insert(0, 'multivideo_analysis')
sys.path.insert(0, '../multivideo_analysis')
import os, sys, re, math, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, spearmanr, wilcoxon
from wordcloud import WordCloud

from pipelines import (build_labeled_audio, build_labels_all, build_seg_all,
                       CANDIDATES, NON_REDUNDANT, CAND_COLOR, extract_candidates)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
FEATURES_DIR = 'Project_Features'

In [ ]:
# ── Audio data: always fast (no visual processing) ───────────────────────────
data = build_labeled_audio()
debate_videos = sorted(v for v in data['video'].unique() if 'vs' in v)

candidates_data = data[
    data['video'].isin(debate_videos) &
    data['speaker'].isin(CANDIDATES)
].copy()
candidates = CANDIDATES          # all 8
color_map  = CAND_COLOR          # consistent palette

KEY_FEATURES = [
    'meanF0Hz','stdevF0Hz','HNR','localJitter','localShimmer',
    'speechrate','articulationrate','npause','asd','f1_mean','f2_mean','fdisp',
]

print(f'{len(data)} audio segments | {len(debate_videos)} debates')
print(f'Candidate segments: {len(candidates_data)}')

In [ ]:
# ── Silhouette quality scores — needed for transcript cells later ─────────────
quality_scores = {}
for v in debate_videos:
    E = np.stack(data[data['video'] == v]['speak_embeddings'].values)
    lbl = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(E)
    quality_scores[v] = silhouette_score(E, lbl, metric='cosine')
print(f'Silhouette scores computed for {len(quality_scores)} debates.')
print(f'Mean silhouette: {np.mean(list(quality_scores.values())):.3f}')

In [ ]:
# ── seg_all: visual pipeline required (slow first run, cached after) ──────────
SEG_ALL_CACHE = 'seg_all.pkl'
LABELS_CACHE  = 'labels_all.pkl'

if os.path.exists(SEG_ALL_CACHE):
    seg_all    = pd.read_pickle(SEG_ALL_CACHE)
    labels_all = pd.read_pickle(LABELS_CACHE) if os.path.exists(LABELS_CACHE) else None
    print(f'Loaded seg_all from cache: {seg_all.shape}')
else:
    print('Building labels_all (visual GMM pipeline, ~15 min)...')
    labels_all = build_labels_all(debate_videos, data)
    labels_all.to_pickle(LABELS_CACHE)
    print('Building seg_all (frame-audio join + aggregation)...')
    seg_all = build_seg_all(debate_videos, data, labels_all)
    seg_all.to_pickle(SEG_ALL_CACHE)
    print(f'seg_all saved: {seg_all.shape}')

emotion_cols      = [c for c in seg_all.columns if c.startswith('prob_')]
pose_feature_cols = ['shoulder_slope','body_openness','head_tilt','wrist_height','torso_height']
seg_cands         = seg_all[seg_all['final_name'].isin(CANDIDATES)].copy()

print(f'seg_all shape: {seg_all.shape}')
print(f'Candidate segments in seg_all: {len(seg_cands)}')
print(f'Emotion cols: {emotion_cols}')
print(f'Pose cols:    {pose_feature_cols}')

---
# Part I — Audio-only multi-video analysis

These sections use only acoustic features and speaker embeddings (`data` from
`build_labeled_audio()`), without any visual information. They mirror the content
that was in `labeling_audio/correlations_audio.ipynb` Parts VI, VII, and VIII.

## I.A — Acoustic feature landscape across all 28 debates

Global distribution and correlation structure of the 12 key acoustic features,
pooled over all ~2600 segments from all debates.

In [ ]:
# Feature distributions — one histogram per key feature
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(KEY_FEATURES):
    axes[i].hist(data[col].dropna(), bins=40, color='steelblue', alpha=0.7, edgecolor='none')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('count')
plt.suptitle('Distribution of key acoustic features (all 28 debates combined)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Full feature correlation matrix — reveals redundant groups
scalar_cols = [c for c in data.columns if c not in
               ['video','time stamp','duration','speak_embeddings','cluster_k3','speaker','party']]
corr = data[scalar_cols].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3, annot_kws={'size': 7}, ax=ax)
ax.set_title('Correlation matrix — all 30 acoustic features (pooled, all debates)', fontsize=13)
plt.tight_layout()
plt.show()

# Top pairs (|r| > 0.4, excluding self-correlations)
seen, rows = set(), []
for (a, b), v in corr.abs().unstack().sort_values(ascending=False).items():
    key = tuple(sorted([a, b]))
    if key not in seen and v < 1.0 and v > 0.4:
        seen.add(key); rows.append({'feature_a': a, 'feature_b': b, 'r': round(corr.loc[a,b],3)})
display(pd.DataFrame(rows))

## I.B — Per-candidate acoustic profiles

How does each candidate's voice differ from the others?
We look at speaking time, turn length, key features, and the normalized profile heatmap.

**Caveat:** speaker labels for Tier-2 debates (Ventura, Pinto, lower silhouette) are
less reliable; treat those two candidates' profiles with some caution.

In [ ]:
# Speaking time and segment counts
print('Segments per candidate:')
print(candidates_data['speaker'].value_counts().to_string())
print()
total_time_all = candidates_data['duration'].sum()
time_share = (candidates_data.groupby('speaker')['duration'].sum()
              / total_time_all * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(time_share.index, time_share.values,
              color=[color_map.get(c,'grey') for c in time_share.index], edgecolor='white')
for bar, val in zip(bars, time_share.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('% of total candidate speaking time')
ax.set_title('Speaking time share per candidate — all 28 debates, host excluded', fontsize=12)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Turn length distributions (violin) and box plots of KEY_FEATURES
fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(data=candidates_data, x='speaker', y='duration',
               order=candidates, palette='tab10', ax=ax, cut=0)
ax.set_xlabel(''); ax.set_ylabel('Segment duration (seconds)')
ax.set_title('Speaking turn duration per candidate', fontsize=12)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, 4, figsize=(18, 12)); axes = axes.flatten()
for i, col in enumerate(KEY_FEATURES):
    plot_data = candidates_data[['speaker', col]].dropna()
    sns.boxplot(data=plot_data, x='speaker', y=col, ax=axes[i],
                palette='tab10', order=candidates, fliersize=2)
    axes[i].set_title(col, fontsize=11); axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=40, labelsize=8)
plt.suptitle('Acoustic features by candidate (host excluded)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Bivariate scatters: speech rate vs articulation, pitch vs HNR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cand in candidates:
    sub = candidates_data[candidates_data['speaker'] == cand]
    axes[0].scatter(sub['speechrate'], sub['articulationrate'],
                    alpha=0.4, s=12, label=cand, color=color_map.get(cand,'grey'))
    axes[1].scatter(sub['meanF0Hz'], sub['HNR'],
                    alpha=0.4, s=12, label=cand, color=color_map.get(cand,'grey'))
axes[0].set_xlabel('speech rate'); axes[0].set_ylabel('articulation rate')
axes[0].set_title('Speech rate vs articulation rate')
axes[0].legend(fontsize=8, bbox_to_anchor=(1.01,1), loc='upper left')
axes[1].set_xlabel('mean pitch (Hz)'); axes[1].set_ylabel('HNR (voice clarity)')
axes[1].set_title('Pitch vs HNR')
plt.suptitle('Bivariate scatter — colored by candidate', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Normalized acoustic profile heatmap — who is high/low on each feature?
profile_nr = candidates_data.groupby('speaker')[NON_REDUNDANT].mean()
profile_norm = pd.DataFrame(
    StandardScaler().fit_transform(profile_nr),
    index=profile_nr.index, columns=profile_nr.columns
)
fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(profile_norm, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.4, annot_kws={'size': 8}, ax=ax)
ax.set_title('Candidate acoustic profiles — z-scored across candidates (red=high, blue=low)', fontsize=12)
plt.tight_layout(); plt.show()

# Mean values table
display(candidates_data.groupby('speaker')[KEY_FEATURES].mean().round(3))

In [ ]:
# PCA of speaker embeddings — do the 8 candidates separate in 512-d space?
emb_matrix = np.stack(candidates_data['speak_embeddings'].values)
pca_emb = PCA(n_components=2, random_state=42)
emb_2d  = pca_emb.fit_transform(emb_matrix)
v1, v2  = pca_emb.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(10, 7))
for cand in candidates:
    mask = candidates_data['speaker'].values == cand
    ax.scatter(emb_2d[mask,0], emb_2d[mask,1],
               label=cand, alpha=0.35, s=12, color=color_map.get(cand,'grey'))
ax.set_title('Speaker embeddings — all 8 candidates (512-d pyannote → 2D PCA)', fontsize=12)
ax.set_xlabel(f'PC1 ({v1:.1f}%)'); ax.set_ylabel(f'PC2 ({v2:.1f}%)')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## I.C — Candidate voice consistency across debates

How stable is each candidate's speaker embedding across their 7 appearances?
Low distance to own global centroid → consistent acoustic identity across opponents and dates.

In [ ]:
# Per-candidate, per-debate centroid; then compare within vs between
cand_debate_rows = []
for (speaker, video), grp in candidates_data.groupby(['speaker','video']):
    E = np.stack(grp['speak_embeddings'].values)
    cand_debate_rows.append({'speaker':speaker,'video':video,
                              'n_segments':len(grp),'total_duration':grp['duration'].sum(),
                              'centroid':E.mean(axis=0)})

cdt = pd.DataFrame(cand_debate_rows)
global_c = cdt.groupby('speaker')['centroid'].apply(
    lambda s: np.mean(np.stack(s.values), axis=0))

per_rows, within_rows = [], []
for speaker, sub in cdt.groupby('speaker'):
    cents    = np.stack(sub['centroid'].values)
    own_c    = global_c[speaker]
    d_own    = cdist(cents, own_c[None,:], metric='cosine').ravel()
    others   = [c for c in global_c.index if c != speaker]
    d_oth    = cdist(cents, np.stack(global_c[others].values), metric='cosine')
    tmp      = sub[['speaker','video','n_segments','total_duration']].copy()
    tmp['dist_to_own']      = d_own
    tmp['nearest_other']    = d_oth.min(axis=1)
    tmp['margin']           = tmp['nearest_other'] - tmp['dist_to_own']
    per_rows.append(tmp)
    if len(cents) > 1:
        pw = cdist(cents, cents, metric='cosine')
        iu = np.triu_indices_from(pw, k=1)
        for d in pw[iu]:
            within_rows.append({'speaker':speaker,'distance':d,'kind':'within-candidate'})

consistency = pd.concat(per_rows, ignore_index=True)
order = consistency.groupby('speaker')['dist_to_own'].mean().sort_values().index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, title in zip(axes,
    ['dist_to_own','margin'],
    ['Distance to own global centroid
(lower = more consistent voice)',
     'Margin vs nearest other candidate
(higher = better separated)']):
    sns.boxplot(data=consistency, x='speaker', y=col, order=order,
                palette=[color_map.get(c,'grey') for c in order], ax=ax, fliersize=0)
    sns.stripplot(data=consistency, x='speaker', y=col, order=order,
                  ax=ax, color='black', alpha=0.6, size=5, jitter=0.15)
    ax.set_title(title, fontsize=11); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
if any(within_rows):
    axes[1].axhline(0, color='gray', ls='--', lw=1)
plt.suptitle('Candidate voice consistency — 512-dim speaker embeddings', fontsize=13, y=1.03)
plt.tight_layout(); plt.show()

display(consistency.groupby('speaker').agg(
    debates=('video','nunique'),
    mean_dist=('dist_to_own','mean'),
    mean_margin=('margin','mean')
).round(4).sort_values('mean_dist'))

## I.D — Transcript analysis across all debates

Lexical analysis of what each candidate actually says: bigrams, richness,
word clouds, and semantic similarity across their debates using 4096-dim EuroLLM
transcript embeddings.

In [ ]:
# Load all speech pkls (debates only)
speech_files = [f for f in os.listdir(FEATURES_DIR) if f.endswith('_speech.pkl')]
sdfs = []
for f in speech_files:
    df = pd.read_pickle(os.path.join(FEATURES_DIR, f))
    df['video'] = f.replace('_speech.pkl','')
    sdfs.append(df)
data_speech = pd.concat(sdfs, ignore_index=True)
data_speech['word_count']   = data_speech['transcript'].apply(lambda t: len(str(t).split()) if pd.notnull(t) else 0)
data_speech['words_per_sec'] = data_speech['word_count'] / data_speech['duration']

# Merge speaker labels
ts_col = 'timestamp' if 'timestamp' in data_speech.columns else 'time stamp'
speech_labeled = data_speech.merge(
    data[['video','time stamp','speaker','party']].rename(columns={'time stamp':ts_col}),
    on=['video',ts_col], how='left')

speech_debates = speech_labeled[speech_labeled['video'].isin(debate_videos)].copy()
print(f'Speech segments: {len(speech_debates)} | labeled: {speech_debates["speaker"].notna().sum()}')

In [ ]:
# Debate structure: segment count vs mean turn length (colored by silhouette quality)
debate_stats = [{'video':v, 'n_segments':len(data[data['video']==v]),
                  'mean_duration':data[data['video']==v]['duration'].mean(),
                  'sil_score':quality_scores.get(v,np.nan)} for v in debate_videos]
stats_df = pd.DataFrame(debate_stats)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(stats_df['n_segments'], stats_df['mean_duration'],
                c=stats_df['sil_score'], cmap='RdYlGn', vmin=0.3, vmax=0.7, s=80)
plt.colorbar(sc, ax=ax, label='silhouette score')
ax.set_xlabel('Number of audio segments'); ax.set_ylabel('Mean segment duration (s)')
ax.set_title('Debate structure: segment count vs turn length
(color = speaker separation quality)')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Word clouds per debate
PT_STOPWORDS = {
    'de','a','o','que','e','do','da','em','um','para','uma','com','nao','por','mais',
    'as','dos','como','mas','foi','ao','ele','das','tem','seu','sua','ou','ser','quando',
    'muito','nos','ja','eu','tambem','so','pelo','pela','ate','isso','ela','entre','era',
    'depois','sem','mesmo','aos','seus','suas','numa','nem','nas','esse','essa','num','ha',
    'sobre','este','esta','foi','pelos','pelas','esses','essas','todo','toda','tudo','isto',
    'aqui','vos','lhes','dele','delas','nela','na','no','ter','sao','boa','noite','entao',
    'assim','porque','temos','agora','bem','vai','pode','vou','acho','anos','dizer','fazer',
    'coisa','coisas','aquilo','portanto','nada','nunca','sempre','ainda','hoje','estou',
    'estamos','estava','vamos','vao','sou','sabe','sei','quer','quero','querem','podemos',
    'deve','falar','faz','fez','disse','diz','ver','dar','estar','sendo','tendo','sido',
    'pessoa','pessoas','gente','lado','parte','forma','vez','caso','ponto','sentido','tempo',
    'momento','exemplo','questao','verdade','nosso','nossa','todos','outro','outros','outra',
    'alguem','ninguem','qualquer','mim','dois',
}

def make_wordcloud(text, title, ax):
    wc = WordCloud(width=600, height=300, background_color='white',
                   stopwords=PT_STOPWORDS, max_words=80, colormap='tab20',
                   min_word_length=3, collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear'); ax.set_title(title, fontsize=9); ax.axis('off')

n = len(debate_videos); ncols = 4; nrows = math.ceil(n/ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows*3.5)); axes = axes.flatten()
for i, v in enumerate(sorted(debate_videos)):
    txt = ' '.join(speech_debates[speech_debates['video']==v]['transcript'].dropna().astype(str))
    make_wordcloud(txt, v.replace('_',' ').replace(' vs ','
vs ')[:30], axes[i])
for j in range(i+1, len(axes)): axes[j].axis('off')
plt.suptitle('Word clouds per debate (stopwords removed)', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Per-candidate bigrams
def tokenize(text, sw=PT_STOPWORDS, ml=3):
    tokens = re.findall(r'[a-zà-ÿ]+(?:-[a-zà-ÿ]+)?',
                        str(text).lower(), flags=re.IGNORECASE)
    return [t for t in tokens if len(t)>=ml and t not in sw]

candidate_text = speech_labeled[
    speech_labeled['video'].isin(debate_videos) &
    speech_labeled['speaker'].isin(CANDIDATES)
].copy()
candidate_text['tokens'] = candidate_text['transcript'].apply(tokenize)

fig, axes = plt.subplots(4, 2, figsize=(16, 18)); axes = axes.flatten()
for ax, cand in zip(axes, sorted(CANDIDATES)):
    sub = candidate_text[candidate_text['speaker']==cand]
    all_tokens = [t for toks in sub['tokens'] for t in toks]
    bigrams = Counter(zip(all_tokens, all_tokens[1:])).most_common(8)
    if not bigrams: ax.axis('off'); continue
    labels  = [' '.join(bg) for bg,_ in bigrams]
    counts  = [c for _,c in bigrams]
    ax.barh(labels[::-1], counts[::-1], color='teal', edgecolor='white')
    ax.set_title(f'{cand.replace("_"," ")}  ({len(all_tokens)} tokens)', fontsize=10)
    ax.set_xlabel('count')
for ax in axes[len(CANDIDATES):]: ax.axis('off')
plt.suptitle('Top transcript bigrams per candidate (stopwords removed)', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Lexical richness per candidate
lex_rows = []
for cand, sub in candidate_text.groupby('speaker'):
    tokens = [t for toks in sub['tokens'] for t in toks]
    counts = Counter(tokens)
    n = len(tokens); u = len(counts); h = sum(1 for c in counts.values() if c==1)
    lex_rows.append({'candidate':cand,'tokens':n,'unique':u,
                     'root_ttr': u/max(n,1)**0.5,'hapax_ratio':h/max(n,1),
                     'mean_words_per_seg': sub['word_count'].mean()})
lex = pd.DataFrame(lex_rows).sort_values('root_ttr', ascending=False)
display(lex.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=lex, x='candidate', y='root_ttr',
            order=lex.sort_values('root_ttr',ascending=False)['candidate'],
            color='slateblue', ax=axes[0])
axes[0].set_title('Lexical richness (root TTR — higher = more varied vocabulary)')
axes[0].tick_params(axis='x', rotation=35)
sns.barplot(data=lex, x='candidate', y='hapax_ratio',
            order=lex.sort_values('hapax_ratio',ascending=False)['candidate'],
            color='steelblue', ax=axes[1])
axes[1].set_title('Hapax ratio (words used only once — higher = more unique words)')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

In [ ]:
# Transcript embedding PCA — do candidates cluster by WHAT they say?
emb_col = 'text_embedding'
speech_cands = candidate_text.dropna(subset=[emb_col]).copy()
speech_cands[emb_col] = speech_cands[emb_col].apply(
    lambda x: x if isinstance(x, np.ndarray) else np.fromstring(str(x).strip('[]'), sep=' '))

T = np.stack(speech_cands[emb_col].values)
pca_t = PCA(n_components=2, random_state=42)
T_2d  = pca_t.fit_transform(PCA(50, random_state=42).fit_transform(T))
v1, v2 = pca_t.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for cand in CANDIDATES:
    mask = speech_cands['speaker'].values == cand
    axes[0].scatter(T_2d[mask,0], T_2d[mask,1], label=cand, alpha=0.35, s=12,
                    color=color_map.get(cand,'grey'))
axes[0].set_title('Transcript embeddings — by candidate')
axes[0].set_xlabel(f'PC1 ({v1:.1f}%)'); axes[0].set_ylabel(f'PC2 ({v2:.1f}%)')
axes[0].legend(fontsize=7, bbox_to_anchor=(1.01,1), loc='upper left')

parties = speech_cands['party'].fillna('host').values
for p in sorted(set(parties)):
    mask = parties==p
    axes[1].scatter(T_2d[mask,0], T_2d[mask,1], label=p, alpha=0.35, s=12)
axes[1].set_title('Transcript embeddings — by party')
axes[1].set_xlabel(f'PC1 ({v1:.1f}%)'); axes[1].set_ylabel(f'PC2 ({v2:.1f}%)')
axes[1].legend(fontsize=7, bbox_to_anchor=(1.01,1), loc='upper left')

plt.suptitle('PCA of 4096-dim EuroLLM transcript embeddings (WHAT was said)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Per-candidate between-debate transcript similarity
# High cosine similarity = candidate keeps same semantic content across different opponents
from scipy.spatial.distance import cdist as _cdist
MIN_SEGS = 5; MONTH_MAP = {'November':11,'December':12,'January':1}

def parse_debate_date_sort(v):
    m = re.search(r'(January|November|December)_(\d+)', v)
    if not m: return (99,99)
    return (MONTH_MAP.get(m.group(1),99), int(m.group(2)))

summary_rows = []
for cand in CANDIDATES:
    sub = speech_cands[speech_cands['speaker']==cand]
    debate_centroids_t = []
    for v, dg in sub.groupby('video'):
        if len(dg) < MIN_SEGS: continue
        c = np.stack(dg[emb_col].values).mean(axis=0)
        debate_centroids_t.append({'video':v,'centroid':c,'n':len(dg)})
    if len(debate_centroids_t) < 2:
        summary_rows.append({'candidate':cand,'debates':len(debate_centroids_t),'mean_sim':np.nan})
        continue
    cents = np.stack([r['centroid'] for r in debate_centroids_t])
    sim   = 1 - _cdist(cents, cents, metric='cosine')
    off   = sim[np.triu_indices_from(sim,k=1)]
    summary_rows.append({'candidate':cand,'debates':len(debate_centroids_t),'mean_sim':off.mean()})

summary_df = pd.DataFrame(summary_rows).sort_values('mean_sim', ascending=False)
display(summary_df.round(3))

fig, ax = plt.subplots(figsize=(10,4))
sns.barplot(data=summary_df.dropna(), x='candidate', y='mean_sim',
            palette=[color_map.get(c,'grey') for c in summary_df.dropna()['candidate']], ax=ax)
ax.set_title('Mean between-debate transcript similarity per candidate
'
             '(higher = more consistent message across opponents)', fontsize=12)
ax.set_xlabel(''); ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

## I.E — Within-debate dynamics: arousal trajectory + vocal entrainment

Averaged across all 28 debates:

In [ ]:
# Arousal trajectory — does debate intensity rise toward the end?
from scipy.stats import pearsonr

AROUSAL_FEATS = ['meanF0Hz','stdevF0Hz','localShimmer','speechrate']
N_BINS = 10

traj_rows = []
for v in debate_videos:
    sub = data[(data['video']==v) & (data['speaker'].isin(CANDIDATES))].copy()
    if len(sub) < N_BINS: continue
    sub = sub.sort_values('time stamp')
    t = sub['time stamp'].values.astype(float)
    sub['progress'] = (t-t.min())/(t.max()-t.min()+1e-9)
    for f in AROUSAL_FEATS:
        x = sub[f].values.astype(float); sd = x.std()
        sub[f+'_z'] = (x-x.mean())/sd if sd>0 else 0.0
    sub['bin'] = np.minimum((sub['progress']*N_BINS).astype(int), N_BINS-1)
    for _,r in sub.iterrows():
        for f in AROUSAL_FEATS: traj_rows.append({'video':v,'bin':r['bin'],'feature':f,'z':r[f+'_z']})

traj = pd.DataFrame(traj_rows)
pivot = traj.groupby(['bin','feature'])['z'].mean().unstack()[AROUSAL_FEATS]

fig, ax = plt.subplots(figsize=(11, 6))
for f in AROUSAL_FEATS: ax.plot(pivot.index, pivot[f], marker='o', label=f)
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.set_xlabel(f'Debate progress (decile 0=start … {N_BINS-1}=end)')
ax.set_ylabel('Mean within-debate z-score')
ax.set_title(f'Arousal trajectory — averaged across {traj["video"].nunique()} debates')
ax.legend(); plt.tight_layout(); plt.show()

for f in AROUSAL_FEATS:
    g = traj[traj['feature']==f].groupby('bin')['z'].mean()
    r,p = pearsonr(g.index.tolist(), g.values.tolist())
    print(f'  {f:16s} r={r:+.3f}  {"p<0.05" if p<0.05 else f"p={p:.2f}"}  -> {"RISES" if r>0 else "FALLS"}')

In [ ]:
# Vocal entrainment — do the two candidates' voices converge within a debate?
from scipy.stats import wilcoxon

ENTRAIN_FEATS = ['meanF0Hz','speechrate','articulationrate','stdevF0Hz']
ent_rows = []
for v in debate_videos:
    cA,cB = extract_candidates(v)
    if cA is None: continue
    sub = data[(data['video']==v) & (data['speaker'].isin([cA,cB]))].copy()
    if len(sub)<8: continue
    sub = sub.sort_values('time stamp')
    t = sub['time stamp'].values.astype(float)
    sub['progress'] = (t-t.min())/(t.max()-t.min()+1e-9)
    scale = sub[ENTRAIN_FEATS].std().values.astype(float); scale[scale==0]=1.0
    def half_means(hdf):
        out={}
        for c in (cA,cB):
            dc=hdf[hdf['speaker']==c]
            if len(dc)==0: return None
            out[c]=dc[ENTRAIN_FEATS].mean().values.astype(float)
        return out
    me=half_means(sub[sub['progress']<0.5]); ml=half_means(sub[sub['progress']>=0.5])
    if me is None or ml is None: continue
    ent_rows.append({'video':v,'dist_early':np.abs((me[cA]-me[cB])/scale).mean(),
                     'dist_late':np.abs((ml[cA]-ml[cB])/scale).mean()})

ent = pd.DataFrame(ent_rows); ent['converged']=ent['dist_late']<ent['dist_early']
fig, ax = plt.subplots(figsize=(10,6))
for _,r in ent.iterrows():
    ax.plot([0,1],[r['dist_early'],r['dist_late']],'-o',
            color='#2ca02c' if r['converged'] else '#d62728', alpha=0.5, lw=1.2)
ax.plot([0,1],[ent['dist_early'].mean(),ent['dist_late'].mean()],
        '-o',color='black',lw=3,markersize=10,label='mean')
ax.set_xticks([0,1]); ax.set_xticklabels(['First half','Second half'])
ax.set_ylabel('Scale-free acoustic distance between candidates')
ax.set_title(f'Vocal entrainment — {len(ent)} debates (green=converged, red=diverged)')
ax.legend(); plt.tight_layout(); plt.show()

n_conv=int(ent['converged'].sum()); n=len(ent)
print(f'Debates: {n} | Converged: {n_conv}/{n} ({100*n_conv/n:.0f}%)')
print(f'Mean dist  early={ent["dist_early"].mean():.3f}  late={ent["dist_late"].mean():.3f}')
if n>=6:
    stat,p=wilcoxon(ent['dist_early'].values,ent['dist_late'].values)
    print(f'Wilcoxon  stat={stat:.1f}  p={p:.3f}  -> {"CONVERGENCE" if ent["dist_late"].mean()<ent["dist_early"].mean() else "DIVERGENCE"}')

---
# Part II — Audio ↔ Visual correlations at multi-video scale

These sections mirror **Sections B, C, D** from
`labeling_visual/corelations_updated.ipynb` (your colleague's single-video analysis),
now run on **`seg_all`** — ~1000+ segments across all 28 debates and all 8 candidates.

The same section letters (B, C, D) are used so you can directly compare single-video
findings with multi-video ones.

## Utility functions (same as in `corelations_updated.ipynb`)

In [ ]:
emotion_colors = {'Anger':'tab:red','Happiness':'tab:olive','Neutral':'tab:gray',
                  'Sadness':'tab:blue','Surprise':'tab:orange','Fear':'tab:purple',
                  'Disgust':'tab:green','Contempt':'tab:brown'}
emotion_order  = ['Anger','Disgust','Contempt','Fear','Sadness','Surprise','Happiness','Neutral']

def heat(cross, title, xlabel='Emotion', ylabel='Audio Feature', figsize=(12,6)):
    plt.figure(figsize=figsize)
    sns.heatmap(cross.astype(float), annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, linewidths=0.5)
    plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.tight_layout(); plt.show()

def within_speaker_corr(df, rows, cols, group='final_name', method='spearman'):
    g = df.dropna(subset=[group])
    out = pd.DataFrame(index=rows, columns=cols, dtype=float)
    for r in rows:
        for c in cols:
            d = g[[r,c,group]].dropna().copy()
            d['rx'] = d[r] - d.groupby(group)[r].transform('mean')
            d['ry'] = d[c] - d.groupby(group)[c].transform('mean')
            out.loc[r,c] = d['rx'].corr(d['ry'], method=method)
    return out

def eta_squared(df, value, factor):
    d = df[[value,factor]].dropna()
    if d[factor].nunique()<2: return np.nan
    grand = d[value].mean()
    ssb = sum(len(g)*(g[value].mean()-grand)**2 for _,g in d.groupby(factor))
    sst = ((d[value]-grand)**2).sum()
    return ssb/sst if sst>0 else np.nan

def corr_pvals(df, rows, cols):
    cc = pd.DataFrame(index=rows, columns=cols, dtype=float)
    pv = pd.DataFrame(index=rows, columns=cols, dtype=float)
    for r in rows:
        for c in cols:
            d = df[[r,c]].dropna()
            rho,p = spearmanr(d[r],d[c])
            cc.loc[r,c]=rho; pv.loc[r,c]=p
    return cc, pv

def heat_sig(cc, pv, title, xlabel='', ylabel='', alpha=0.05, figsize=(14,6)):
    ann = pd.DataFrame(index=cc.index, columns=cc.columns)
    for r in cc.index:
        for c in cc.columns:
            ann.loc[r,c] = f'{cc.loc[r,c]:.2f}
*' if pv.loc[r,c]<=alpha else f'{cc.loc[r,c]:.2f}
(ns)'
    plt.figure(figsize=figsize)
    sns.heatmap(cc.astype(float), annot=ann, fmt='', cmap='coolwarm', vmin=-1, vmax=1,
                linewidths=0.5, annot_kws={'size':8})
    plt.title(f'{title}
(* = p<{alpha}, ns = not significant)')
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.tight_layout(); plt.show()

print(f'seg_cands ready: {len(seg_cands)} segments | emotion cols: {len(emotion_cols)}')

## B. Audio ↔ Visual  (segment level — one row per audio segment with a visible speaking face)

Each segment has: 10 acoustic features + 8 emotion probabilities + 5 pose features.
At **single-video scale** (`corelations_updated`) there were ~50 segments and 2 candidates.
Here we have **1000+ segments and 8 candidates** — correlations that were noise before
may now be statistically real.

### B1. Audio ↔ Emotion (all candidates combined)

In [ ]:
cc = seg_cands[NON_REDUNDANT + emotion_cols].corr(method='spearman').loc[NON_REDUNDANT, emotion_cols]
heat(cc, f'Audio vs Emotion — all candidates combined (n={len(seg_cands)} segments)')

### B2. Audio ↔ Emotion, per candidate

**The multi-video payoff:** with 8 candidates we can now see whether the
audio-emotion relationship is the same for everyone, or whether some candidates
show a unique pattern (e.g. Ventura's pitch vs emotion may differ from Filipe's).

In [ ]:
n_cands = len(CANDIDATES)
ncols = 2; nrows = math.ceil(n_cands/ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 6*nrows))
axes = axes.flatten()
for ax, cand in zip(axes, CANDIDATES):
    s = seg_cands[seg_cands['final_name']==cand]
    if len(s) < 10:
        ax.axis('off'); ax.set_title(f'{cand} (n<10)'); continue
    cc = s[NON_REDUNDANT + emotion_cols].corr(method='spearman').loc[NON_REDUNDANT, emotion_cols]
    sns.heatmap(cc.astype(float), annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, linewidths=0.5, ax=ax, cbar=False, annot_kws={'size':7})
    ax.set_title(f'{cand.replace("_"," ")} (n={len(s)})', fontsize=11)
for ax in axes[n_cands:]: ax.axis('off')
plt.suptitle('Audio vs Emotion — per candidate (8 matrices side by side)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### B3. Audio feature distributions by dominant emotion

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9)); axes = axes.flatten()
for i, feat in enumerate(NON_REDUNDANT):
    sns.boxplot(data=seg_cands, x='top_emotion', y=feat, order=emotion_order,
                palette=emotion_colors, hue='top_emotion', legend=False,
                ax=axes[i], linewidth=0.8, fliersize=2)
    axes[i].set_title(feat); axes[i].set_xlabel(''); axes[i].set_ylabel('')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8); axes[i].grid(axis='y', alpha=0.3)
fig.suptitle(f'Audio feature distributions by dominant emotion — all {len(seg_cands)} segments', fontsize=14)
plt.tight_layout(); plt.show()

### B4. Mean audio per dominant emotion (z-scored)

In [ ]:
from scipy.stats import zscore as _zscore
summary_z = seg_cands.groupby('top_emotion')[NON_REDUNDANT].mean().apply(_zscore)
plt.figure(figsize=(12,6))
sns.heatmap(summary_z, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Mean audio features per dominant emotion (z-scored across emotions)')
plt.tight_layout(); plt.show()
print(seg_cands.groupby('top_emotion')[NON_REDUNDANT].mean().round(2))

### B5. Audio ↔ Pose (speaking face only)

In [ ]:
valid_pose = seg_cands.dropna(subset=pose_feature_cols)
if len(valid_pose) > 0:
    cc = valid_pose[pose_feature_cols + NON_REDUNDANT].corr(method='spearman').loc[pose_feature_cols, NON_REDUNDANT]
    heat(cc, f'Pose vs Audio — {len(valid_pose)} segments with valid pose',
         xlabel='Audio Feature', ylabel='Pose Feature', figsize=(14,5))
else:
    print('No segments with valid pose features in seg_cands')

## C. Is the signal real, or just *who* is speaking?

**At single-video scale** (2 candidates, ~50 segments) within-speaker correlations
were near zero — the pooled signal was driven almost entirely by between-candidate
differences. With **8 candidates and 1000+ segments** we now have the power to
test this properly.

### C1. Within-speaker correlation (identity controlled)

Audio↔emotion after subtracting each candidate's mean. If values collapse toward 0
relative to B1, the pooled correlation was pure between-candidate variance,
not a genuine within-person acoustic-emotion link.

In [ ]:
wsc = within_speaker_corr(seg_cands, NON_REDUNDANT, emotion_cols, group='final_name')
heat(wsc, 'Audio vs Emotion — WITHIN speaker (identity controlled, all 8 candidates)')

### C2. Variance partition (η²): explained by *who* vs *which emotion*

**The key multi-video question:** for each acoustic feature, how much of its variance
comes from *who is speaking* (candidate identity) vs *what emotion they show*?
With only 2 candidates the η² test was underpowered. With 8 candidates it is meaningful.

In [ ]:
var_part = pd.DataFrame({
    'by_candidate': {f: eta_squared(seg_cands, f, 'final_name') for f in NON_REDUNDANT},
    'by_emotion':   {f: eta_squared(seg_cands, f, 'top_emotion') for f in NON_REDUNDANT},
}).round(3)
print(var_part)

var_part.plot(kind='bar', figsize=(12,5), color=['#4C72B0','#DD8452'], edgecolor='white')
plt.axhline(0.14, color='gray', ls='--', lw=1, label='medium effect (η²=0.14)')
plt.ylabel('η² (proportion of variance explained)')
plt.title('Variance partition: how much is explained by CANDIDATE vs EMOTION?')
plt.legend(); plt.xticks(rotation=35, ha='right')
plt.tight_layout(); plt.show()

### C3. Significance (Spearman with p-values, segment level)

In [ ]:
cc, pv = corr_pvals(seg_cands, NON_REDUNDANT, emotion_cols)
heat_sig(cc, pv, f'Audio vs Emotion (n={len(seg_cands)} segments)',
         xlabel='Emotion', ylabel='Audio Feature')

if len(valid_pose) > 10:
    cc2, pv2 = corr_pvals(valid_pose, pose_feature_cols, NON_REDUNDANT)
    heat_sig(cc2, pv2, 'Pose vs Audio (segment level, significant cells marked)',
             xlabel='Audio Feature', ylabel='Pose Feature')

## D. Is one modality enough?

CCA (Canonical Correlation Analysis) measures how much the audio space and the
visual (emotion+pose) space actually share. Low canonical correlations → they are
largely independent, meaning **no single modality is enough** to characterise the debates.

At **single-video scale** (~50 rows) CCA was statistically underpowered. With **1000+
independent segments** the test is now valid.

In [ ]:
visual_features = pose_feature_cols + emotion_cols
audio_features  = NON_REDUNDANT
d = seg_cands[visual_features + audio_features].dropna()
print(f'Segments used for CCA: {len(d)}')

# PCA structure per modality
mods = {'Visual (pose+emotion)': visual_features,
        'Audio':                 audio_features,
        'All combined':          visual_features + audio_features}
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (nm, fs) in zip(axes, mods.items()):
    X2 = PCA(2, random_state=42).fit_transform(StandardScaler().fit_transform(d[fs]))
    ax.scatter(X2[:,0], X2[:,1], c='steelblue', alpha=0.4, s=15, edgecolors='none')
    ax.set_title(nm); ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.grid(alpha=0.3)
plt.suptitle('PCA structure per modality (segments)'); plt.tight_layout(); plt.show()

# CCA
Xv = StandardScaler().fit_transform(d[visual_features])
Xa = StandardScaler().fit_transform(d[audio_features])
ncomp = min(3, Xv.shape[1], Xa.shape[1])
Uv, Ua = CCA(n_components=ncomp).fit(Xv, Xa).transform(Xv, Xa)
print('\nCanonical correlations (visual vs audio):')
for i in range(ncomp):
    r,p = pearsonr(Uv[:,i], Ua[:,i])
    print(f'  component {i+1}: r={r:.3f}  p={p:.4f}')
print('\nLow canonical correlations → modalities carry different information → one modality is NOT enough.')

---
# Part III — New findings only possible at multi-video scale

These analyses require aggregating across all 28 debates and cannot be done
on a single video. They are the **headline Part 3 results**.

## III.1 — Per-candidate emotion profile

Who is consistently **angry, surprised, sad** across all their debates?
For each candidate we compute their mean emotion probability profile (averaged over
all segments from all their debates). This is only possible with `seg_all` covering
all 28 debates.

In [ ]:
emotion_profile = seg_cands.groupby('final_name')[emotion_cols].mean()
# rename columns: prob_Anger -> Anger
emotion_profile.columns = [c.replace('prob_','') for c in emotion_profile.columns]
emotion_profile = emotion_profile.reindex(CANDIDATES)

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(emotion_profile, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, annot_kws={'size':9})
ax.set_title('Mean emotion probability per candidate across all debates\n'
             '(higher = that emotion appears more often while this candidate speaks)', fontsize=12)
ax.set_ylabel('Candidate')
plt.tight_layout(); plt.show()

print('Dominant emotion per candidate:')
print(emotion_profile.idxmax(axis=1).to_string())
print()
print('Least common emotion per candidate:')
print(emotion_profile.idxmin(axis=1).to_string())

In [ ]:
# Radar / spider chart of emotion profiles per candidate
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

emotions = [c.replace('prob_','') for c in emotion_cols]
N = len(emotions)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

fig, axes = plt.subplots(2, 4, figsize=(20, 10), subplot_kw=dict(projection='polar'))
axes = axes.flatten()

for ax, cand in zip(axes, CANDIDATES):
    if cand not in emotion_profile.index:
        ax.axis('off'); continue
    vals = list(emotion_profile.loc[cand].values) + [emotion_profile.loc[cand].values[0]]
    ax.plot(angles, vals, 'o-', lw=2, color=CAND_COLOR.get(cand,'grey'))
    ax.fill(angles, vals, alpha=0.2, color=CAND_COLOR.get(cand,'grey'))
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(emotions, fontsize=9)
    ax.set_title(cand.replace('_',' '), fontsize=11, pad=15,
                 color=CAND_COLOR.get(cand,'grey'), fontweight='bold')
    ax.set_ylim(0, emotion_profile.values.max()*1.1)

for ax in axes[len(CANDIDATES):]: ax.axis('off')
plt.suptitle('Emotion profile per candidate — radar chart', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## III.2 — Poll correlation

Can we predict a candidate's poll standing from their acoustic and emotional features?
We align each debate to the nearest poll in `polls_timeline.pkl` and compute
Spearman correlations between candidate-level features and their poll percentage.

In [ ]:
polls = pd.read_pickle(os.path.join(FEATURES_DIR, 'polls_timeline.pkl'))

MONTHS_MAP = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
              'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}

def parse_date_str(s):
    parts = str(s).split('_')
    if len(parts)!=2: return None
    mon = MONTHS_MAP.get(parts[0])
    if not mon: return None
    yr = 2025 if mon>=9 else 2026
    return pd.Timestamp(yr, mon, int(parts[1]))

def parse_video_date(v):
    m = re.search(r'(January|February|March|April|May|June|July|August|'
                  r'September|October|November|December)_(\d+)', v)
    if not m: return None
    mon = MONTHS_MAP[m.group(1)]; day = int(m.group(2))
    yr = 2025 if mon>=10 else 2026
    return pd.Timestamp(yr, mon, day)

polls['poll_date'] = polls['date'].apply(parse_date_str)
polls = polls.dropna(subset=['poll_date']).sort_values('poll_date')

# Per-debate, per-candidate: mean acoustic + emotion features
seg_all['debate_date'] = seg_all['video'].apply(parse_video_date)
feat_cols = NON_REDUNDANT + [c for c in emotion_cols if c in seg_all.columns]

debate_cand_feats = (seg_all[seg_all['final_name'].isin(CANDIDATES)]
                     .groupby(['video','final_name','debate_date'])[feat_cols]
                     .mean().reset_index())

# For each debate, find the nearest poll
def nearest_poll_value(cand, debate_dt):
    if debate_dt is None or cand not in polls.columns: return np.nan
    diffs = (polls['poll_date'] - debate_dt).abs()
    idx = diffs.idxmin()
    return polls.loc[idx, cand]

debate_cand_feats['poll_pct'] = debate_cand_feats.apply(
    lambda r: nearest_poll_value(r['final_name'], r['debate_date']), axis=1)

print(f'Rows for correlation: {len(debate_cand_feats.dropna(subset=["poll_pct"]))}')
display(debate_cand_feats.head())

In [ ]:
# Spearman correlation: acoustic/emotion features vs poll percentage
valid = debate_cand_feats.dropna(subset=['poll_pct'])
poll_corr = {}
for f in feat_cols:
    d = valid[[f,'poll_pct']].dropna()
    if len(d)<10: continue
    rho,p = spearmanr(d[f], d['poll_pct'])
    poll_corr[f] = {'rho':rho,'p':p,'n':len(d)}

poll_corr_df = (pd.DataFrame(poll_corr).T
                .sort_values('rho', key=abs, ascending=False))
print('Feature correlations with poll standing:')
display(poll_corr_df.round(3).head(15))

# Bar chart of top correlations
top = poll_corr_df.dropna().head(12)
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#2ca02c' if r>0 else '#d62728' for r in top['rho']]
bars = ax.barh(top.index, top['rho'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
for bar, (_, row) in zip(bars, top.iterrows()):
    sig = '*' if row['p']<0.05 else ''
    ax.text(row['rho'], bar.get_y()+bar.get_height()/2,
            f' {sig}', va='center', fontsize=10, color='black')
ax.set_xlabel('Spearman ρ with poll %  (* = p<0.05)')
ax.set_title('Which acoustic/emotion features correlate with poll standing?\n'
             '(positive = feature is higher for better-polling candidates)')
plt.tight_layout(); plt.show()

## III.3 — Candidate-opponent effect

Does a candidate **sound or look different** depending on who they are debating?
We build a candidate × opponent matrix for key features: if Candidate A's pitch
is systematically higher when debating Candidate B than C, that is an opponent effect.
This is only testable across multiple debates.

In [ ]:
# Build candidate × opponent feature means
opp_rows = []
for v in debate_videos:
    cA,cB = extract_candidates(v)
    if cA is None: continue
    for cand,opp in [(cA,cB),(cB,cA)]:
        segs = seg_all[(seg_all['video']==v) & (seg_all['final_name']==cand)]
        if len(segs)==0: continue
        row = {'candidate':cand,'opponent':opp,'n_segs':len(segs)}
        row.update(segs[NON_REDUNDANT].mean().to_dict())
        if emotion_cols:
            row.update(segs[emotion_cols].mean().to_dict())
        opp_rows.append(row)

opp_df = pd.DataFrame(opp_rows)
print(f'Candidate-opponent records: {len(opp_df)}')

# For each candidate: show how pitch and speechrate vary by opponent
SHOW_FEATS = ['meanF0Hz','speechrate','HNR']
fig, axes = plt.subplots(1, len(SHOW_FEATS), figsize=(18, 6))
for ax, feat in zip(axes, SHOW_FEATS):
    pivot = opp_df.pivot_table(index='candidate', columns='opponent', values=feat, aggfunc='mean')
    pivot = pivot.reindex(index=CANDIDATES, columns=CANDIDATES)
    sns.heatmap(pivot.astype(float), annot=True, fmt='.1f', cmap='RdBu_r',
                center=pivot.stack().mean(), ax=ax, linewidths=0.4, cbar=False,
                annot_kws={'size':8})
    ax.set_title(f'{feat}\n(row=candidate, col=opponent)', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.suptitle('Candidate × Opponent effect: does a candidate sound different\n'
             'depending on who they debate?', fontsize=13, y=1.03)
plt.tight_layout(); plt.show()

## III.4 — Joint dimensionality reduction

PCA on the full 23-feature space (10 acoustic + 8 emotion + 5 pose) across all
candidate segments. Do the first few components separate candidates, debates,
or emotions? This reveals the most important axes of variation in the multi-modal
feature space.

In [ ]:
all_features = NON_REDUNDANT + emotion_cols + pose_feature_cols
joint = seg_cands.dropna(subset=all_features).copy()
print(f'Segments for joint PCA: {len(joint)} | Features: {len(all_features)}')

X_scaled = StandardScaler().fit_transform(joint[all_features])

# Scree plot
pca_full = PCA(random_state=42).fit(X_scaled)
cum_var  = np.cumsum(pca_full.explained_variance_ratio_) * 100
fig, ax  = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(cum_var)+1), cum_var, marker='o', ms=4)
ax.axhline(80, color='red', ls='--', lw=1, label='80% threshold')
ax.set_xlabel('Number of components'); ax.set_ylabel('Cumulative variance (%)')
ax.set_title('Scree plot — 23-feature joint space'); ax.legend()
plt.tight_layout(); plt.show()
n80 = np.argmax(cum_var>=80)+1
print(f'{n80} components explain ≥80% of variance')

In [ ]:
# 2D PCA scatter — colored by candidate, top emotion, party
X_2d = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
v1_j, v2_j = PCA(n_components=2, random_state=42).fit(X_scaled).explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# by candidate
for cand in CANDIDATES:
    mask = joint['final_name'].values == cand
    axes[0].scatter(X_2d[mask,0], X_2d[mask,1], label=cand, alpha=0.4, s=15,
                    color=CAND_COLOR.get(cand,'grey'))
axes[0].set_title('By candidate'); axes[0].legend(fontsize=7)

# by dominant emotion
for em in emotion_order:
    mask = joint['top_emotion'].values == em
    axes[1].scatter(X_2d[mask,0], X_2d[mask,1], label=em, alpha=0.4, s=15,
                    color=emotion_colors.get(em,'grey'))
axes[1].set_title('By dominant emotion'); axes[1].legend(fontsize=7)

# by party
parties_j = joint['final_name'].map({c: seg_all[seg_all['final_name']==c]['final_name'].iloc[0]
                                      if len(seg_all[seg_all['final_name']==c])>0
                                      else c for c in CANDIDATES})
for _, g in joint.groupby('final_name'):
    mask = joint['final_name'].values == g['final_name'].iloc[0]
    axes[2].scatter(X_2d[mask,0], X_2d[mask,1], label=g['final_name'].iloc[0],
                    alpha=0.35, s=12, color=CAND_COLOR.get(g['final_name'].iloc[0],'grey'))
axes[2].set_title('By candidate (for party reference)'); axes[2].legend(fontsize=7)

for ax in axes:
    ax.set_xlabel(f'PC1 ({v1_j:.1f}%)'); ax.set_ylabel(f'PC2 ({v2_j:.1f}%)')
plt.suptitle('Joint PCA — 23-feature space (acoustic + emotion + pose)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Feature loadings on PC1 and PC2 — what drives the main axes of variation?
pca2 = PCA(n_components=2, random_state=42).fit(X_scaled)
loadings = pd.DataFrame(pca2.components_.T, index=all_features, columns=['PC1','PC2'])
loadings['abs_pc1'] = loadings['PC1'].abs()
loadings['abs_pc2'] = loadings['PC2'].abs()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, pc in zip(axes, ['PC1','PC2']):
    top = loadings[pc].abs().nlargest(10)
    colors = ['#2ca02c' if loadings.loc[f,pc]>0 else '#d62728' for f in top.index]
    ax.barh(top.index[::-1], loadings.loc[top.index[::-1], pc].values,
            color=colors[::-1], edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel(f'Loading on {pc}')
    ax.set_title(f'Top 10 features driving {pc}\n(green=positive, red=negative loading)')
plt.tight_layout(); plt.show()